# Run one job on Colab

This notebook holds no model or data code. It checks out the repository and runs one job file from `configs/jobs/` with the repository's own command. The job file names the data to fetch and the experiments to run. Each run's manifest, scores, forecasts and checkpoint are saved to Google Drive.

Before running: choose a GPU under Runtime > Change runtime type. If the job uses M5, add a Kaggle API token to Colab's Secrets panel and put the secret's name in `KAGGLE_SECRET` below.

In [ ]:
JOB = "configs/jobs/frat_seeds.yaml"  # job files to run, in order, separated by spaces
REF = "main"  # branch or commit of the repository to run
OUT = "/content/drive/MyDrive/retail-world-models/results"  # where results are saved
KAGGLE_SECRET = "KAGGLE"  # name of the Colab secret that holds the Kaggle API token
RELEASE_WHEN_DONE = False  # hand the GPU back when the job ends, so an idle session does not use compute units

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get(KAGGLE_SECRET)
except Exception:
    print("No Kaggle secret available. Jobs that download M5 will stop at the download.")

In [ ]:
%cd /content
!rm -rf retail-world-models
!git clone -q https://github.com/kar488/retail-world-models
%cd /content/retail-world-models
!git checkout -q {REF} && git log -1 --format="running commit %H"
!pip install -q -e ".[trees,deep,fetch]"

In [ ]:
!nvidia-smi -L
!python -m rwm.experiments.job --job {JOB} --out "{OUT}"

if RELEASE_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()  # make sure every result file has reached Drive first
    runtime.unassign()